# 02 Performance Comparison

This notebook reproduces the cortex performance comparison for imputation and clustering. It embeds the small metric helpers from `utils.py`, so the notebook is self-contained.

In [ ]:
from __future__ import annotations

import os
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyro
import torch
from sklearn.cluster import KMeans
from sklearn.decomposition import FactorAnalysis
from sklearn.metrics import adjusted_rand_score as ARI
from sklearn.metrics import normalized_mutual_info_score as NMI
from sklearn.metrics import silhouette_score
from sklearn.model_selection import train_test_split

from FA import FA
from ZIFA import ZIFA

warnings.filterwarnings("ignore")

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "datasets/cortex/datasets"

SEED = 2026
K = 10
N_REPEAT = 10
N_CELLTYPE = 7

INIT_EPOCHS_IMPUTATION = 2000
INIT_EPOCHS_CLUSTERING = 1000
MAX_EPOCHS = 100000

np.random.seed(SEED)
torch.manual_seed(SEED)

print("Project:", PROJECT_DIR)
print("Data dir:", DATA_DIR)
print("CUDA available:", torch.cuda.is_available())


In [ ]:
def cluster_scores(latent_space, K, labels_true):
    labels_pred = KMeans(K, n_init=20, random_state=SEED).fit_predict(latent_space)
    return [
        silhouette_score(latent_space, labels_true),
        NMI(labels_true, labels_pred),
        ARI(labels_true, labels_pred),
    ]


def imputation_error(X_mean, X, X_zero, i, j, ix):
    all_index = i[ix], j[ix]
    x, y = X_mean[all_index], X[all_index]
    return np.median(np.abs(x - y))


In [ ]:
def load_cortex_data():
    expression_path = DATA_DIR / "expression_mRNA_17-Aug-2014.txt"
    if not expression_path.exists():
        raise FileNotFoundError(f"Missing cortex expression file: {expression_path}")

    X = pd.read_csv(expression_path, sep="	", low_memory=False).T
    clusters = np.array(X[7], dtype=str)[2:]
    _, labels = np.unique(clusters, return_inverse=True)

    X = X.loc[:, 10:]
    X = X.drop(X.index[0])
    expression = np.array(X, dtype=int)[1:]

    selected = np.std(expression, axis=0).argsort()[-558:][::-1]
    expression = expression[:, selected]

    X_train, X_test, label_train, label_test = train_test_split(
        expression, labels, random_state=0
    )

    rng = np.random.default_rng(SEED)
    X_zero_counts = np.array(X_train, dtype=np.float32, copy=True)
    i, j = np.nonzero(X_zero_counts)
    ix = rng.choice(np.arange(len(i)), int(np.floor(0.1 * len(i))), replace=False)
    X_zero_counts[i[ix], j[ix]] = 0.0

    return {
        "X_true_counts": np.asarray(X_train, dtype=np.float32),
        "X_zero_counts": X_zero_counts,
        "X_zero_log": np.log1p(X_zero_counts).astype(np.float32),
        "expression_train_log": np.log1p(np.asarray(X_train, dtype=np.float32)),
        "label_train": np.asarray(label_train),
        "i": i,
        "j": j,
        "ix": ix,
    }


data = load_cortex_data()
X_true_counts = data["X_true_counts"]
X_zero_counts = data["X_zero_counts"]
X_zero = data["X_zero_log"]
expression_train = data["expression_train_log"]
label_train = data["label_train"]
i = data["i"]
j = data["j"]
ix = data["ix"]
N, D = X_zero.shape

print("X_zero:", X_zero.shape)
print("expression_train:", expression_train.shape)
print("labels:", np.unique(label_train), "n_celltype=", len(np.unique(label_train)))
print("Held-out nonzero entries:", len(ix))


## Shared Experiment Helpers

In [ ]:
def reset_random_state(seed: int) -> None:
    np.random.seed(seed)
    torch.manual_seed(seed)
    pyro.clear_param_store()


def fa_display_name(method: str) -> str:
    return {"FA": "FA", "VI": "FA-VI", "Amortized": "FA-AVI"}[method]


def zifa_display_name(method: str) -> str:
    return {"ZIFA": "ZIFA", "VI": "ZIFA-VI", "Amortized": "ZIFA-AVI"}[method]


def initialize_loss_threshold(family: str, X: np.ndarray, epochs: int) -> float:
    reset_random_state(SEED)
    if family == "FA":
        model = FA(K=K, method="amortized", loss_threshold=float("-inf"), max_epochs=epochs)
    elif family == "ZIFA":
        model = ZIFA(K=K, method="pyro", inference="amortized", loss_threshold=float("-inf"), max_epochs=epochs)
    else:
        raise ValueError(f"Unknown family: {family}")
    model.fit(X)
    loss = float(model.result["loss"])
    threshold = float(np.floor(loss))
    print(f"{family} initial loss after {epochs} epochs: {loss:.4f}; threshold={threshold:.4f}")
    return threshold


def dense_reconstruction(result: dict, D: int) -> np.ndarray:
    A = result["A"]
    Z = result["latent"]
    mu = result.get("mu", result.get("mus"))
    mu = np.asarray(mu).reshape([1, -1])
    return np.exp(Z @ A.T + mu.reshape([1, A.shape[0]])) - 1


## Imputation Experiments

Runs all methods for imputation and records median absolute imputation error plus runtime.


In [ ]:
def run_fa_imputation(loss_threshold: float) -> dict:
    results = {
        "FA": {"error": [], "time": []},
        "VI": {"error": [], "time": []},
        "Amortized": {"error": [], "time": []},
    }

    print("Running FA imputation...")
    for repeat in range(N_REPEAT):
        start = time.time()
        alg = FactorAnalysis(n_components=K, random_state=repeat)
        alg.fit(X_zero)
        Z = alg.transform(X_zero)
        X_impu = np.exp(Z @ alg.components_ + alg.mean_.reshape([1, D])) - 1
        err = imputation_error(X_impu, X_true_counts, X_zero_counts, i, j, ix)
        elapsed = time.time() - start
        results["FA"]["error"].append(float(err))
        results["FA"]["time"].append(float(elapsed))
        print(f"  FA repeat {repeat + 1}: error={err:.6f}, time={elapsed:.2f}s")

    print("Running FA-VI and FA-AVI imputation...")
    for repeat in range(N_REPEAT):
        for method, inference in [("VI", "vi"), ("Amortized", "amortized")]:
            reset_random_state(1234 + repeat)
            start = time.time()
            model = FA(K=K, method=inference, loss_threshold=loss_threshold, max_epochs=MAX_EPOCHS)
            model.fit(X_zero)
            X_impu = dense_reconstruction(model.result, D)
            err = imputation_error(X_impu, X_true_counts, X_zero_counts, i, j, ix)
            elapsed = time.time() - start
            results[method]["error"].append(float(err))
            results[method]["time"].append(float(elapsed))
            print(f"  {fa_display_name(method)} repeat {repeat + 1}: error={err:.6f}, time={elapsed:.2f}s")

    return results


def run_zifa_imputation(loss_threshold: float) -> dict:
    results = {
        "ZIFA": {"error": [], "time": []},
        "VI": {"error": [], "time": []},
        "Amortized": {"error": [], "time": []},
    }

    non_zero_cols = ~(np.abs(X_zero) < 1e-6).all(axis=0)
    zero_cols = ~non_zero_cols
    X_zero_filtered = X_zero[:, non_zero_cols]

    print("Running ZIFA imputation...")
    for repeat in range(N_REPEAT):
        reset_random_state(1234 + repeat)
        start = time.time()
        model = ZIFA(K=K, method="classic")
        model.fit(X_zero_filtered)
        X_impu_filtered = dense_reconstruction(model.result, X_zero_filtered.shape[1])
        X_impu = np.zeros_like(X_true_counts, dtype=np.float64)
        X_impu[:, non_zero_cols] = X_impu_filtered
        X_impu[:, zero_cols] = 0.0
        err = imputation_error(X_impu, X_true_counts, X_zero_counts, i, j, ix)
        elapsed = time.time() - start
        results["ZIFA"]["error"].append(float(err))
        results["ZIFA"]["time"].append(float(elapsed))
        print(f"  ZIFA repeat {repeat + 1}: error={err:.6f}, time={elapsed:.2f}s")

    print("Running ZIFA-VI and ZIFA-AVI imputation...")
    for repeat in range(N_REPEAT):
        for method, inference in [("VI", "vi"), ("Amortized", "amortized")]:
            reset_random_state(1234 + repeat)
            start = time.time()
            model = ZIFA(K=K, method="pyro", inference=inference, loss_threshold=loss_threshold, max_epochs=MAX_EPOCHS)
            model.fit(X_zero)
            X_impu = dense_reconstruction(model.result, D)
            err = imputation_error(X_impu, X_true_counts, X_zero_counts, i, j, ix)
            elapsed = time.time() - start
            results[method]["error"].append(float(err))
            results[method]["time"].append(float(elapsed))
            print(f"  {zifa_display_name(method)} repeat {repeat + 1}: error={err:.6f}, time={elapsed:.2f}s")

    return results


In [ ]:
fa_imputation_threshold = initialize_loss_threshold("FA", X_zero, INIT_EPOCHS_IMPUTATION)
zifa_imputation_threshold = initialize_loss_threshold("ZIFA", X_zero, INIT_EPOCHS_IMPUTATION)

results_imputation_fa = run_fa_imputation(fa_imputation_threshold)
results_imputation_zifa = run_zifa_imputation(zifa_imputation_threshold)


In [ ]:
def imputation_summary(results_fa: dict, results_zifa: dict) -> pd.DataFrame:
    rows = []
    for method, values in results_fa.items():
        rows.append({
            "Method": fa_display_name(method),
            "Metric": "Imputation Error",
            "Mean": np.mean(values["error"]),
            "Median": np.median(values["error"]),
            "Std": np.std(values["error"], ddof=1),
            "Mean time (s)": np.mean(values.get("time", [np.nan])),
        })
    for method, values in results_zifa.items():
        rows.append({
            "Method": zifa_display_name(method),
            "Metric": "Imputation Error",
            "Mean": np.mean(values["error"]),
            "Median": np.median(values["error"]),
            "Std": np.std(values["error"], ddof=1),
            "Mean time (s)": np.mean(values.get("time", [np.nan])),
        })
    df = pd.DataFrame(rows)
    return df

imputation_summary_df = imputation_summary(results_imputation_fa, results_imputation_zifa)
imputation_summary_df


## Clustering Experiments

Runs all methods for clustering on the log1p training expression matrix and records ASW, NMI, ARI, and runtime.

In [ ]:
def run_fa_clustering(loss_threshold: float) -> dict:
    results = {
        "FA": {"ASW": [], "NMI": [], "ARI": [], "time": []},
        "VI": {"ASW": [], "NMI": [], "ARI": [], "time": []},
        "Amortized": {"ASW": [], "NMI": [], "ARI": [], "time": []},
    }

    print("Running FA clustering...")
    for repeat in range(N_REPEAT):
        start = time.time()
        alg = FactorAnalysis(n_components=K, random_state=repeat)
        alg.fit(expression_train)
        latent = alg.transform(expression_train)
        elapsed = time.time() - start
        scores = cluster_scores(latent, N_CELLTYPE, label_train)
        for metric, value in zip(["ASW", "NMI", "ARI"], scores):
            results["FA"][metric].append(float(value))
        results["FA"]["time"].append(float(elapsed))
        print(f"  FA repeat {repeat + 1}: ASW={scores[0]:.4f}, NMI={scores[1]:.4f}, ARI={scores[2]:.4f}, time={elapsed:.2f}s")

    print("Running FA-VI and FA-AVI clustering...")
    for repeat in range(N_REPEAT):
        for method, inference in [("VI", "vi"), ("Amortized", "amortized")]:
            reset_random_state(1234 + repeat)
            start = time.time()
            model = FA(K=K, method=inference, loss_threshold=loss_threshold, max_epochs=MAX_EPOCHS)
            model.fit(expression_train)
            elapsed = time.time() - start
            scores = cluster_scores(model.latent, N_CELLTYPE, label_train)
            for metric, value in zip(["ASW", "NMI", "ARI"], scores):
                results[method][metric].append(float(value))
            results[method]["time"].append(float(elapsed))
            print(f"  {fa_display_name(method)} repeat {repeat + 1}: ASW={scores[0]:.4f}, NMI={scores[1]:.4f}, ARI={scores[2]:.4f}, time={elapsed:.2f}s")

    return results


def run_zifa_clustering(loss_threshold: float) -> dict:
    results = {
        "ZIFA": {"ASW": [], "NMI": [], "ARI": [], "time": []},
        "VI": {"ASW": [], "NMI": [], "ARI": [], "time": []},
        "Amortized": {"ASW": [], "NMI": [], "ARI": [], "time": []},
    }

    print("Running ZIFA clustering...")
    for repeat in range(N_REPEAT):
        reset_random_state(1234 + repeat)
        start = time.time()
        model = ZIFA(K=K, method="classic")
        model.fit(expression_train)
        elapsed = time.time() - start
        scores = cluster_scores(model.latent, N_CELLTYPE, label_train)
        for metric, value in zip(["ASW", "NMI", "ARI"], scores):
            results["ZIFA"][metric].append(float(value))
        results["ZIFA"]["time"].append(float(elapsed))
        print(f"  ZIFA repeat {repeat + 1}: ASW={scores[0]:.4f}, NMI={scores[1]:.4f}, ARI={scores[2]:.4f}, time={elapsed:.2f}s")

    print("Running ZIFA-VI and ZIFA-AVI clustering...")
    for repeat in range(N_REPEAT):
        for method, inference in [("VI", "vi"), ("Amortized", "amortized")]:
            reset_random_state(1234 + repeat)
            start = time.time()
            model = ZIFA(K=K, method="pyro", inference=inference, loss_threshold=loss_threshold, max_epochs=MAX_EPOCHS)
            model.fit(expression_train)
            elapsed = time.time() - start
            scores = cluster_scores(model.latent, N_CELLTYPE, label_train)
            for metric, value in zip(["ASW", "NMI", "ARI"], scores):
                results[method][metric].append(float(value))
            results[method]["time"].append(float(elapsed))
            print(f"  {zifa_display_name(method)} repeat {repeat + 1}: ASW={scores[0]:.4f}, NMI={scores[1]:.4f}, ARI={scores[2]:.4f}, time={elapsed:.2f}s")

    return results


In [ ]:
fa_clustering_threshold = initialize_loss_threshold("FA", expression_train, INIT_EPOCHS_CLUSTERING)
zifa_clustering_threshold = initialize_loss_threshold("ZIFA", expression_train, INIT_EPOCHS_CLUSTERING)

results_clustering_fa = run_fa_clustering(fa_clustering_threshold)
results_clustering_zifa = run_zifa_clustering(zifa_clustering_threshold)


In [ ]:
def clustering_summary(results_fa: dict, results_zifa: dict) -> pd.DataFrame:
    rows = []
    for method, values in results_fa.items():
        for metric in ["ASW", "NMI", "ARI"]:
            rows.append({
                "Method": fa_display_name(method),
                "Metric": metric,
                "Mean": np.mean(values[metric]),
                "Median": np.median(values[metric]),
                "Std": np.std(values[metric], ddof=1),
                "Mean time (s)": np.mean(values.get("time", [np.nan])),
            })
    for method, values in results_zifa.items():
        for metric in ["ASW", "NMI", "ARI"]:
            rows.append({
                "Method": zifa_display_name(method),
                "Metric": metric,
                "Mean": np.mean(values[metric]),
                "Median": np.median(values[metric]),
                "Std": np.std(values[metric], ddof=1),
                "Mean time (s)": np.mean(values.get("time", [np.nan])),
            })
    df = pd.DataFrame(rows)
    return df

clustering_summary_df = clustering_summary(results_clustering_fa, results_clustering_zifa)
clustering_summary_df


## Plotting

In [ ]:
METHOD_ORDER = ["FA", "FA-VI", "FA-AVI", "ZIFA", "ZIFA-VI", "ZIFA-AVI"]
DEFAULT_COLORS = plt.rcParams["axes.prop_cycle"].by_key()["color"]
FA_COLOR = DEFAULT_COLORS[0]
ZIFA_COLOR = DEFAULT_COLORS[1]
METHOD_COLORS = {method: (FA_COLOR if method.startswith("FA") else ZIFA_COLOR) for method in METHOD_ORDER}


def metric_dataframe(results_fa: dict, results_zifa: dict, metric: str) -> pd.DataFrame:
    rows = []
    for method, values in results_fa.items():
        display = fa_display_name(method)
        key = "error" if metric == "Imputation Error" else metric
        for value in values[key]:
            rows.append({"Method": display, "Value": value, "Family": "FA"})
    for method, values in results_zifa.items():
        display = zifa_display_name(method)
        key = "error" if metric == "Imputation Error" else metric
        for value in values[key]:
            rows.append({"Method": display, "Value": value, "Family": "ZIFA"})
    df = pd.DataFrame(rows)
    df["Method"] = pd.Categorical(df["Method"], categories=METHOD_ORDER, ordered=True)
    return df.sort_values("Method")


def plot_metric(df: pd.DataFrame, ylabel: str, ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(6.0, 4.2))

    grouped = [df.loc[df["Method"] == method, "Value"].to_numpy() for method in METHOD_ORDER]
    positions = np.arange(1, len(METHOD_ORDER) + 1)
    box = ax.boxplot(grouped, positions=positions, widths=0.55, patch_artist=True, showfliers=False)

    for patch, method in zip(box["boxes"], METHOD_ORDER):
        patch.set_facecolor(METHOD_COLORS[method])
        patch.set_alpha(0.65)
        patch.set_edgecolor(METHOD_COLORS[method])
    for median in box["medians"]:
        median.set_color("white")
        median.set_linewidth(1.8)
    for whisker in box["whiskers"]:
        whisker.set_color("0.35")
    for cap in box["caps"]:
        cap.set_color("0.35")

    rng = np.random.default_rng(SEED)
    for pos, method, values in zip(positions, METHOD_ORDER, grouped):
        x = rng.normal(pos, 0.045, size=len(values))
        ax.scatter(x, values, s=18, color=METHOD_COLORS[method], alpha=0.85, edgecolor="none")

    ax.set_xticks(positions)
    ax.set_xticklabels(METHOD_ORDER, rotation=45, ha="right")
    ax.set_ylabel(ylabel)
    ax.set_xlabel("")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.yaxis.grid(True, linestyle="-", linewidth=0.5, color="0.88")
    ax.set_axisbelow(True)
    return ax


In [ ]:
df_imputation = metric_dataframe(results_imputation_fa, results_imputation_zifa, "Imputation Error")
ax = plot_metric(df_imputation, "Imputation Error")
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, metric in zip(axes, ["ARI", "ASW", "NMI"]):
    df_metric = metric_dataframe(results_clustering_fa, results_clustering_zifa, metric)
    plot_metric(df_metric, metric, ax=ax)
fig.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4.2))
plot_metric(df_imputation, "Imputation Error", ax=axes[0])
for ax, metric in zip(axes[1:], ["ARI", "ASW", "NMI"]):
    df_metric = metric_dataframe(results_clustering_fa, results_clustering_zifa, metric)
    plot_metric(df_metric, metric, ax=ax)
fig.tight_layout()
plt.show()
